# Post-Hoc Flow Preconditioning Experiments

Cache-first notebook for the quadratic sanity check and the fixed tiny-MLP weight-space experiment. The main probe map is function-space residual geometry on a fixed probe batch, with optional theta damping for the MLP protocol.

In [ ]:
from __future__ import annotations

import json
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.flow_preconditioning import ExperimentConfig, run_or_load
from post_train_research.loss_landscape_analysis.flow_preconditioning.config import torch_dtype
from post_train_research.loss_landscape_analysis.flow_preconditioning.reporting import geometry_summary, paired_deltas
from post_train_research.loss_landscape_analysis.flow_preconditioning.toy_mlp import MLPRegressionProblem, mlp_forward

plt.rcParams['figure.dpi'] = 130

cfg = ExperimentConfig(
    run_label='paperish',
    artifact_root='artifacts/loss_landscape_analysis/flow_preconditioning',
    cache_first=True,
    force_rerun=False,
    seeds=(0, 1, 2, 3, 4),
    k_tune=8,
    k_eval=32,
    budgets=(300, 1000),
    rho_values=(0.0, 1e-3, 1e-2, 5e-2),
    main_rho=1e-2,
)

tables = run_or_load(cfg)
figure_dir = tables.output_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)

print('output_dir:', tables.output_dir)
print('results:', tables.results.shape)
print('curves:', tables.curves.shape)
print('geometry:', tables.geometry.shape)
print('selected_lrs:', tables.selected_lrs.shape)
print('figures:', tables.figure_paths)


## Tables

In [ ]:
display(Markdown('### Selected learning rates'))
display(tables.selected_lrs.sort_values(['experiment', 'seed', 'rho', 'budget', 'method']).head(40))

display(Markdown('### Aggregate metrics'))
display(tables.aggregate.sort_values(['experiment', 'rho', 'budget', 'optimizer', 'method']).head(80))

geom_summary = geometry_summary(tables.geometry)
display(Markdown('### Geometry summary'))
display(geom_summary.sort_values(['experiment', 'rho', 'coordinate', 'seed']).head(80))

delta_table = paired_deltas(tables.results)
display(Markdown('### Paired AULC deltas'))
display(delta_table.groupby(['experiment', 'rho', 'budget', 'optimizer', 'comparison'])['delta_aulc'].median().reset_index().head(80))


## Sanity Check

In [ ]:
max_budget = int(max(cfg.budgets))
sanity_curves = tables.curves[
    (tables.curves['experiment'] == 'sanity')
    & (tables.curves['split'] == 'eval')
    & (tables.curves['budget'] == max_budget)
]

fig, ax = plt.subplots(figsize=(8, 5))
for method, frame in sanity_curves.groupby('method'):
    median = frame.groupby('step')['train_loss'].median()
    ax.plot(median.index, median.values, label=method)
ax.set_yscale('log')
ax.set_xlabel('step')
ax.set_ylabel('median train loss')
ax.set_title('Sanity quadratic optimization')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
fig.savefig(figure_dir / 'sanity_optimization_curves.png', dpi=160, bbox_inches='tight')
plt.show()

display(tables.aggregate[tables.aggregate['experiment'] == 'sanity'].sort_values(['budget', 'optimizer', 'method']))
display(geom_summary[geom_summary['experiment'] == 'sanity'].sort_values(['coordinate', 'seed']))


## Main MLP Experiment

In [ ]:
main_curves = tables.curves[
    (tables.curves['experiment'] == 'mlp')
    & (tables.curves['split'] == 'eval')
    & (tables.curves['rho'] == float(cfg.main_rho))
    & (tables.curves['budget'] == max_budget)
]
show_methods = ['direct_sgd', 'trained_flow_sgd', 'random_flow_sgd', 'direct_adam', 'trained_flow_adam', 'random_flow_adam']

fig, ax = plt.subplots(figsize=(9, 5))
for method in show_methods:
    frame = main_curves[main_curves['method'] == method]
    if frame.empty:
        continue
    grouped = frame.groupby('step')['train_loss']
    median = grouped.median()
    q25 = grouped.quantile(0.25)
    q75 = grouped.quantile(0.75)
    ax.plot(median.index, median.values, label=method)
    ax.fill_between(median.index, q25.values, q75.values, alpha=0.12)
ax.set_yscale('log')
ax.set_xlabel('step')
ax.set_ylabel('train loss')
ax.set_title(f'MLP eval curves, rho={cfg.main_rho:g}, T={max_budget}')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8, ncol=2)
fig.savefig(figure_dir / 'mlp_median_loss_curves.png', dpi=160, bbox_inches='tight')
plt.show()

scatter = delta_table[
    (delta_table['experiment'] == 'mlp')
    & (delta_table['rho'] == float(cfg.main_rho))
    & (delta_table['budget'] == max_budget)
    & (delta_table['comparison'] == 'trained_minus_direct')
]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for optimizer, frame in scatter.groupby('optimizer'):
    ax.scatter(frame['right_aulc'], frame['left_aulc'], s=22, alpha=0.7, label=optimizer)
lo = float(np.nanmin([scatter['right_aulc'].min(), scatter['left_aulc'].min()]))
hi = float(np.nanmax([scatter['right_aulc'].max(), scatter['left_aulc'].max()]))
ax.plot([lo, hi], [lo, hi], color='black', linewidth=1)
ax.set_xlabel('direct AULC')
ax.set_ylabel('trained-flow AULC')
ax.set_title('Matched-start paired AULC')
ax.grid(True, alpha=0.25)
ax.legend()
fig.savefig(figure_dir / 'mlp_paired_aulc_scatter.png', dpi=160, bbox_inches='tight')
plt.show()

display(tables.aggregate[(tables.aggregate['experiment'] == 'mlp') & (tables.aggregate['budget'] == max_budget)].sort_values(['rho', 'optimizer', 'method']))


## Geometry Diagnostics

In [ ]:
main_geom = geom_summary[(geom_summary['experiment'] == 'mlp') & (geom_summary['rho'] == float(cfg.main_rho))]
display(main_geom.sort_values(['coordinate', 'seed']))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
main_geom.boxplot(column='isometry_objective', by='coordinate', ax=axes[0])
main_geom.boxplot(column='median_log_condition_number', by='coordinate', ax=axes[1])
axes[0].set_title('Heldout R')
axes[1].set_title('Heldout median log cond')
for ax in axes:
    ax.set_xlabel('coordinate')
    ax.grid(True, alpha=0.25)
fig.suptitle('')
fig.savefig(figure_dir / 'mlp_geometry_diagnostics.png', dpi=160, bbox_inches='tight')
plt.show()

eig_rows = tables.geometry[(tables.geometry['experiment'] == 'mlp') & (tables.geometry['rho'] == float(cfg.main_rho))]
eig_values = []
for row in eig_rows.itertuples(index=False):
    for value in json.loads(row.eigenvalues_json):
        eig_values.append({'coordinate': row.coordinate, 'eigenvalue': value})
eig_df = pd.DataFrame(eig_values)
fig, ax = plt.subplots(figsize=(8, 4))
for coordinate, frame in eig_df.groupby('coordinate'):
    ax.hist(np.log10(np.maximum(frame['eigenvalue'].to_numpy(dtype=float), 1e-30)), bins=60, alpha=0.45, label=coordinate)
ax.set_xlabel('log10 eigenvalue')
ax.set_ylabel('count')
ax.set_title('Pullback metric eigenvalues')
ax.legend()
fig.savefig(figure_dir / 'mlp_geometry_eigenvalues.png', dpi=160, bbox_inches='tight')
plt.show()


## Representative Predictions

In [ ]:
eval_results = tables.results[
    (tables.results['experiment'] == 'mlp')
    & (tables.results['split'] == 'eval')
    & (~tables.results['candidate'])
    & (tables.results['rho'] == float(cfg.main_rho))
    & (tables.results['budget'] == max_budget)
]
pairs = eval_results[eval_results['method'].isin(['direct_adam', 'trained_flow_adam'])]
chosen_key = None
for key, frame in pairs.groupby(['seed', 'start_index']):
    if {'direct_adam', 'trained_flow_adam'}.issubset(set(frame['method'])):
        chosen_key = key
        break
if chosen_key is None:
    raise RuntimeError('No matched direct_adam/trained_flow_adam eval pair found')
seed, start_index = chosen_key
plot_cfg = replace(cfg, device='cpu', dtype='float32')
problem = MLPRegressionProblem(plot_cfg, seed=int(seed), device=torch.device('cpu'), dtype=torch.float32)
x = problem.x_test.detach().cpu()
target = problem.y_test.detach().cpu()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x.numpy(), target.numpy(), color='black', linewidth=2, label='target')
for method, frame in pairs[(pairs['seed'] == seed) & (pairs['start_index'] == start_index)].groupby('method'):
    theta = torch.tensor(json.loads(frame.iloc[0]['final_theta_json']), dtype=torch.float32)
    pred = mlp_forward(theta, x).detach().cpu()
    ax.plot(x.numpy(), pred.numpy(), label=f'{method} final')
ax.set_title(f'Representative predictions: seed={seed}, start={start_index}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.25)
ax.legend()
fig.savefig(figure_dir / 'mlp_representative_predictions.png', dpi=160, bbox_inches='tight')
plt.show()


## Automatic Interpretation

In [ ]:
display(Markdown(tables.interpretation_markdown))
print('interpretation saved to:', tables.output_dir / 'interpretation.md')
